# Research API Decision Framework (Phase 0.2)

This notebook is the **routing-policy baseline** for Agentic Research follow-up behavior.
It is intentionally separate from `research_api_investigation.ipynb`, which focuses on API/provider capability checks.

It answers:
1. Which provider performs best by question difficulty (easy/medium/hard)?
2. When should the agent call web research at all (vs static context only)?
3. When should we do a **single API call** vs a **mini-research tool loop**?
4. Can difficulty and action type be auto-classified by an LLM/rules?
5. What architecture/settings/cost-latency trade-offs should be used in production?
6. Does an end-to-end ticker walkthrough (CB) support the policy?

---

## Output artifacts
- `data/research_eval_questions_12.json` (benchmark dataset)
- `data/research_eval_results_12.json` (raw provider outputs)
- `data/research_eval_scores_12.json` (scored outputs)
- `data/research_policy_recommendation.json` (final recommended policy)

---

## Run order
1. Setup
2. Dataset (12 questions)
3. Provider benchmark run
4. Scoring and comparison
5. Gating + action policy
6. Architecture and settings plan
7. CB end-to-end test

In [1]:
import os
import re
import sys
import time
import json
from pathlib import Path
from datetime import datetime

import httpx
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version}")
print(f"Timestamp    : {datetime.now().isoformat()}")

# Optional project settings (safe fallback if not importable)
try:
    from src.utils.config import get_settings
    settings = get_settings()
    SETTINGS_AVAILABLE = True
except Exception as e:
    settings = None
    SETTINGS_AVAILABLE = False
    print(f"[WARN] Could not load project settings: {e}")

Project root : /home/deploy_invest_ai/investment-agent
Python       : 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
Timestamp    : 2026-03-16T20:53:33.775642


In [2]:
# 12-question benchmark dataset: 4 easy, 4 medium, 4 hard

QUESTION_SET = [
    # Easy (direct fact lookups)
    {
        "id": "E1",
        "difficulty": "easy",
        "member": "strategy",
        "ticker": "AAPL",
        "question": "AAPL FY2025 Q4 total revenue and YoY growth from official release",
        "ground_truth": {
            "answer": "Revenue was $102.5B, up 8% YoY.",
            "key_facts": ["$102.5B", "8%"],
            "sources": ["https://www.apple.com/newsroom/"]
        },
        "expected_action": "single_api_call",
    },
    {
        "id": "E2",
        "difficulty": "easy",
        "member": "strategy",
        "ticker": "AAPL",
        "question": "AAPL FY2025 Q4 Services revenue and YoY growth",
        "ground_truth": {
            "answer": "Services revenue was $28.8B, up 15% YoY.",
            "key_facts": ["$28.8B", "15%"],
            "sources": ["https://www.apple.com/newsroom/"]
        },
        "expected_action": "single_api_call",
    },
    {
        "id": "E3",
        "difficulty": "easy",
        "member": "risk",
        "ticker": "AAPL",
        "question": "Apple CIK and canonical data.sec.gov submissions endpoint",
        "ground_truth": {
            "answer": "CIK 320193, endpoint CIK0000320193.json",
            "key_facts": ["320193", "CIK0000320193.json"],
            "sources": ["https://www.sec.gov/files/company_tickers.json"]
        },
        "expected_action": "single_api_call",
    },
    {
        "id": "E4",
        "difficulty": "easy",
        "member": "risk",
        "ticker": "GENERAL",
        "question": "SEC annual vs quarterly filing forms",
        "ground_truth": {
            "answer": "10-K annual, 10-Q quarterly",
            "key_facts": ["10-K", "10-Q"],
            "sources": ["https://www.sec.gov/forms"]
        },
        "expected_action": "single_api_call",
    },

    # Medium (comparison + context)
    {
        "id": "M1",
        "difficulty": "medium",
        "member": "skeptic",
        "ticker": "CB",
        "question": "CB valuation vs peers: is premium justified by underwriting quality?",
        "ground_truth": {
            "answer": "Needs valuation multiple vs peer quality context; not a single fact.",
            "key_facts": ["valuation", "peers", "underwriting"],
            "sources": ["https://www.sec.gov/forms"]
        },
        "expected_action": "single_api_call",
    },
    {
        "id": "M2",
        "difficulty": "medium",
        "member": "skeptic",
        "ticker": "CB",
        "question": "Recent analyst downgrades/upgrades for CB and key rationale",
        "ground_truth": {
            "answer": "Should include both direction and rationale summary.",
            "key_facts": ["analyst", "upgrade", "downgrade", "rationale"],
            "sources": ["https://www.sec.gov/forms"]
        },
        "expected_action": "single_api_call",
    },
    {
        "id": "M3",
        "difficulty": "medium",
        "member": "risk",
        "ticker": "GENERAL",
        "question": "8-K filing deadline and when it is triggered",
        "ground_truth": {
            "answer": "8-K for material events, generally due within 4 business days.",
            "key_facts": ["8-K", "material events", "4 business days"],
            "sources": ["https://www.sec.gov/forms"]
        },
        "expected_action": "single_api_call",
    },
    {
        "id": "M4",
        "difficulty": "medium",
        "member": "strategy",
        "ticker": "CB",
        "question": "CB catastrophe exposure trend and implications for forward earnings",
        "ground_truth": {
            "answer": "Requires synthesis of cat exposure + earnings outlook.",
            "key_facts": ["catastrophe", "exposure", "forward earnings"],
            "sources": ["https://www.sec.gov/forms"]
        },
        "expected_action": "single_api_call",
    },

    # Hard (multi-hop / event mapping)
    {
        "id": "H1",
        "difficulty": "hard",
        "member": "risk",
        "ticker": "GENERAL",
        "question": "Event mapping: CFO resigns and insider buys shares in same week; list required SEC forms and timing",
        "ground_truth": {
            "answer": "CFO resignation -> 8-K; insider buy -> Form 4 (typically 2 business days).",
            "key_facts": ["8-K", "Form 4", "2 business days"],
            "sources": ["https://www.sec.gov/forms"]
        },
        "expected_action": "mini_research",
    },
    {
        "id": "H2",
        "difficulty": "hard",
        "member": "skeptic",
        "ticker": "CB",
        "question": "Construct bear-case chain: earnings miss risk, reserve adequacy risk, valuation de-rating pathway",
        "ground_truth": {
            "answer": "Requires linked multi-point downside chain with evidence.",
            "key_facts": ["earnings miss", "reserve adequacy", "de-rating"],
            "sources": ["https://www.sec.gov/forms"]
        },
        "expected_action": "mini_research",
    },
    {
        "id": "H3",
        "difficulty": "hard",
        "member": "strategy",
        "ticker": "CB",
        "question": "Bull vs bear synthesis for CB: what evidence would materially flip HOLD to BUY or SELL?",
        "ground_truth": {
            "answer": "Requires decision-threshold logic with opposing evidence.",
            "key_facts": ["flip conditions", "bull evidence", "bear evidence"],
            "sources": ["https://www.sec.gov/forms"]
        },
        "expected_action": "mini_research",
    },
    {
        "id": "H4",
        "difficulty": "hard",
        "member": "strategy",
        "ticker": "GENERAL",
        "question": "Build filing timeline after quarter-end including 10-Q, 10-K, and event-driven 8-K interactions",
        "ground_truth": {
            "answer": "Needs integrated timeline reasoning across filing types.",
            "key_facts": ["10-Q", "10-K", "8-K", "timeline"],
            "sources": ["https://www.sec.gov/forms"]
        },
        "expected_action": "mini_research",
    },
]

questions_path = PROJECT_ROOT / "data" / "research_eval_questions_12.json"
questions_path.parent.mkdir(parents=True, exist_ok=True)
questions_path.write_text(json.dumps(QUESTION_SET, indent=2), encoding="utf-8")

questions_df = pd.DataFrame(QUESTION_SET)
print("Question dataset summary:")
display(questions_df[["id", "difficulty", "member", "ticker", "expected_action", "question"]])
print("Counts by difficulty:")
print(questions_df["difficulty"].value_counts())

Question dataset summary:


,id,difficulty,member,ticker,expected_action,question
0,E1,easy,strategy,AAPL,single_api_call,AAPL FY2025 Q4 total revenue and YoY growth fr...
1,E2,easy,strategy,AAPL,single_api_call,AAPL FY2025 Q4 Services revenue and YoY growth
2,E3,easy,risk,AAPL,single_api_call,Apple CIK and canonical data.sec.gov submissio...
3,E4,easy,risk,GENERAL,single_api_call,SEC annual vs quarterly filing forms
4,M1,medium,skeptic,CB,single_api_call,CB valuation vs peers: is premium justified by...
5,M2,medium,skeptic,CB,single_api_call,Recent analyst downgrades/upgrades for CB and ...
6,M3,medium,risk,GENERAL,single_api_call,8-K filing deadline and when it is triggered
7,M4,medium,strategy,CB,single_api_call,CB catastrophe exposure trend and implications...
8,H1,hard,risk,GENERAL,mini_research,Event mapping: CFO resigns and insider buys sh...
9,H2,hard,skeptic,CB,mini_research,"Construct bear-case chain: earnings miss risk,..."


Counts by difficulty:
difficulty
easy      4
medium    4
hard      4
Name: count, dtype: int64


In [3]:
# Provider clients

def call_brave_search(query: str, count: int = 5) -> dict:
    key = os.environ.get("BRAVE_SEARCH_API_KEY")
    if not key:
        return {"error": "BRAVE_SEARCH_API_KEY not set", "latency_ms": 0}
    url = "https://api.search.brave.com/res/v1/web/search"
    headers = {"X-Subscription-Token": key.strip()}
    t0 = time.perf_counter()
    try:
        resp = httpx.get(url, params={"q": query, "count": count}, headers=headers, timeout=20)
        latency_ms = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency_ms}
        data = resp.json()
        results = data.get("web", {}).get("results", [])
        snippets = [{"title": r.get("title", ""), "text": (r.get("description", "") or "")[:220]} for r in results[:count]]
        return {"ok": True, "latency_ms": latency_ms, "num_results": len(results), "answer": "", "snippets": snippets}
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}


def call_brave_answers(query: str, max_retries: int = 1) -> dict:
    key = os.environ.get("BRAVE_ANSWER_API_KEY")
    if not key:
        return {"error": "BRAVE_ANSWER_API_KEY not set", "latency_ms": 0}
    url = "https://api.search.brave.com/res/v1/chat/completions"
    headers = {"Content-Type": "application/json", "X-Subscription-Token": key.strip()}
    body = {"model": "brave", "stream": False, "messages": [{"role": "user", "content": query}]}

    total_latency = 0.0
    last_error = "unknown"
    for attempt in range(max_retries + 1):
        t0 = time.perf_counter()
        try:
            resp = httpx.post(url, json=body, headers=headers, timeout=45)
            total_latency += (time.perf_counter() - t0) * 1000
            if resp.status_code in (408, 429, 500, 502, 503, 504) and attempt < max_retries:
                time.sleep(1.2 * (attempt + 1))
                continue
            if resp.status_code != 200:
                return {"error": f"HTTP {resp.status_code}: {resp.text[:160]}", "latency_ms": total_latency}
            data = resp.json()
            choices = data.get("choices", []) or []
            ans = (((choices[0] or {}).get("message") or {}).get("content") or "")[:400] if choices else ""
            return {"ok": True, "latency_ms": total_latency, "num_results": 0, "answer": ans, "snippets": []}
        except Exception as e:
            total_latency += (time.perf_counter() - t0) * 1000
            last_error = str(e)
            if attempt < max_retries:
                time.sleep(1.2 * (attempt + 1))
                continue
    return {"error": last_error, "latency_ms": total_latency}


def call_tavily_search(query: str, max_results: int = 5) -> dict:
    key = os.environ.get("TAVILY_API_KEY")
    if not key:
        return {"error": "TAVILY_API_KEY not set", "latency_ms": 0}
    url = "https://api.tavily.com/search"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {key.strip()}"}
    body = {"query": query, "search_depth": "basic", "topic": "finance", "max_results": max_results, "include_answer": True}
    t0 = time.perf_counter()
    try:
        resp = httpx.post(url, json=body, headers=headers, timeout=30)
        latency_ms = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency_ms}
        data = resp.json()
        results = data.get("results", [])
        snippets = [{"title": r.get("title", ""), "text": (r.get("content", "") or "")[:220]} for r in results[:max_results]]
        answer = (data.get("answer") or "")[:400]
        return {"ok": True, "latency_ms": latency_ms, "num_results": len(results), "answer": answer, "snippets": snippets}
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}


def run_provider(provider: str, query: str) -> dict:
    if provider == "brave_search":
        return call_brave_search(query)
    if provider == "brave_answers":
        return call_brave_answers(query)
    if provider == "tavily":
        return call_tavily_search(query)
    return {"error": f"Unknown provider {provider}", "latency_ms": 0}

In [4]:
# Benchmark run across all 12 questions and 3 providers

PROVIDERS = ["brave_search", "brave_answers", "tavily"]
# Safer default when reopening notebook: avoid accidental API spend.
RUN_PROVIDER_CALLS = False

results = []
if RUN_PROVIDER_CALLS:
    for q in QUESTION_SET:
        for p in PROVIDERS:
            out = run_provider(p, q["question"])
            results.append({
                "id": q["id"],
                "difficulty": q["difficulty"],
                "member": q["member"],
                "ticker": q["ticker"],
                "question": q["question"],
                "expected_action": q["expected_action"],
                "provider": p,
                **out,
            })
            print(f"[{p}] {q['id']} {('PASS' if out.get('ok') else 'FAIL')} {out.get('latency_ms', 0):.0f}ms")

results_path = PROJECT_ROOT / "data" / "research_eval_results_12.json"
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

results_df = pd.DataFrame(results)
print("\nRaw benchmark outputs:")
display(results_df[["id", "difficulty", "provider", "ok", "latency_ms", "num_results"]])

[brave_search] E1 PASS 951ms
[brave_answers] E1 PASS 3441ms
[tavily] E1 PASS 2256ms
[brave_search] E2 PASS 991ms
[brave_answers] E2 PASS 2653ms
[tavily] E2 PASS 2244ms
[brave_search] E3 PASS 1081ms
[brave_answers] E3 PASS 11157ms
[tavily] E3 PASS 2112ms
[brave_search] E4 PASS 1018ms
[brave_answers] E4 PASS 19292ms
[tavily] E4 PASS 2189ms
[brave_search] M1 PASS 984ms
[brave_answers] M1 PASS 15498ms
[tavily] M1 PASS 2370ms
[brave_search] M2 PASS 976ms
[brave_answers] M2 PASS 6958ms
[tavily] M2 PASS 2364ms
[brave_search] M3 PASS 982ms
[brave_answers] M3 PASS 21705ms
[tavily] M3 PASS 2497ms
[brave_search] M4 PASS 973ms
[brave_answers] M4 PASS 22208ms
[tavily] M4 PASS 2461ms
[brave_search] H1 PASS 990ms
[brave_answers] H1 PASS 37535ms
[tavily] H1 PASS 3253ms
[brave_search] H2 PASS 1034ms
[brave_answers] H2 PASS 30520ms
[tavily] H2 PASS 3417ms
[brave_search] H3 PASS 981ms
[brave_answers] H3 PASS 18971ms
[tavily] H3 PASS 3253ms
[brave_search] H4 PASS 943ms
[brave_answers] H4 PASS 82018ms
[tav

,id,difficulty,provider,ok,latency_ms,num_results
0,E1,easy,brave_search,True,951.015531,5
1,E1,easy,brave_answers,True,3441.292471,0
2,E1,easy,tavily,True,2255.934250,5
3,E2,easy,brave_search,True,991.460929,5
4,E2,easy,brave_answers,True,2653.488980,0
5,E2,easy,tavily,True,2243.649511,5
6,E3,easy,brave_search,True,1081.406682,5
7,E3,easy,brave_answers,True,11156.561350,0
8,E3,easy,tavily,True,2112.163618,1
9,E4,easy,brave_search,True,1018.101737,5


In [5]:
# Scoring: heuristic ground-truth match + quality/noise signals

def _norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").lower()).strip()


def _response_text(row: dict) -> str:
    parts = [row.get("answer", "")]
    for s in row.get("snippets", []) or []:
        parts.append((s.get("title", "") + " " + s.get("text", ""))[:260])
    return _norm(" ".join(parts))


def _fact_coverage(text: str, facts: list[str]) -> float:
    if not facts:
        return 0.0
    hits = sum(1 for f in facts if _norm(f) in text)
    return hits / len(facts)


def _noise_score(text: str) -> float:
    # 1(best/low noise) -> 5(high noise)
    noise_terms = ["not financial advice", "insufficient context", "unable to", "i cannot", "uncertain"]
    hits = sum(1 for t in noise_terms if t in text)
    return min(5.0, 1.0 + hits * 1.2)


def score_row(row: dict, qmeta: dict) -> dict:
    txt = _response_text(row)
    facts = qmeta["ground_truth"].get("key_facts", [])

    fact_cov = _fact_coverage(txt, facts)
    relevance = 5 if any(_norm(k) in txt for k in [_norm(qmeta["ticker"]), _norm(qmeta["question"][:25])]) else 3
    evidence_quality = 2 + min(3, len((row.get("snippets") or [])))
    financial_signal = 2 + min(3, sum(1 for t in ["earnings", "guidance", "risk", "valuation", "filing"] if t in txt))
    noise_level = _noise_score(txt)
    truth_alignment = max(1, round(fact_cov * 5))

    overall = round((relevance + financial_signal + evidence_quality + truth_alignment + (6 - noise_level)) / 5)
    return {
        "relevance": int(relevance),
        "financial_signal": int(financial_signal),
        "evidence_quality": int(evidence_quality),
        "truth_alignment": int(truth_alignment),
        "noise_level": float(round(noise_level, 2)),
        "overall_quality": int(overall),
        "fact_coverage": round(fact_cov, 3),
    }


q_by_id = {q["id"]: q for q in QUESTION_SET}
scored = []
for r in results:
    s = score_row(r, q_by_id[r["id"]])
    scored.append({**r, **s})

scores_df = pd.DataFrame(scored)

scores_path = PROJECT_ROOT / "data" / "research_eval_scores_12.json"
scores_path.write_text(json.dumps(scored, indent=2), encoding="utf-8")

print("Scored outputs (sample):")
display(scores_df[["id", "difficulty", "provider", "overall_quality", "truth_alignment", "noise_level", "latency_ms"]].head(18))

print("Provider avg by difficulty:")
provider_diff = (
    scores_df.groupby(["difficulty", "provider"], as_index=False)
    .agg(overall_quality=("overall_quality", "mean"), truth_alignment=("truth_alignment", "mean"), latency_ms=("latency_ms", "mean"))
    .sort_values(["difficulty", "overall_quality"], ascending=[True, False])
)
display(provider_diff.round(2))

Scored outputs (sample):


,id,difficulty,provider,overall_quality,truth_alignment,noise_level,latency_ms
0,E1,easy,brave_search,5,5,1.0,951.015531
1,E1,easy,brave_answers,3,1,1.0,3441.292471
2,E1,easy,tavily,4,2,1.0,2255.934250
3,E2,easy,brave_search,5,5,1.0,991.460929
4,E2,easy,brave_answers,3,1,1.0,2653.488980
5,E2,easy,tavily,4,2,1.0,2243.649511
6,E3,easy,brave_search,3,1,1.0,1081.406682
7,E3,easy,brave_answers,3,5,1.0,11156.561350
8,E3,easy,tavily,3,2,1.0,2112.163618
9,E4,easy,brave_search,5,5,1.0,1018.101737


Provider avg by difficulty:


,difficulty,provider,overall_quality,truth_alignment,latency_ms
1,easy,brave_search,4.50,4.00,1010.50
2,easy,tavily,3.75,2.75,2200.15
0,easy,brave_answers,3.25,3.00,9135.73
5,hard,tavily,4.25,3.25,3111.46
4,hard,brave_search,3.75,1.50,987.32
3,hard,brave_answers,3.25,2.50,42261.03
8,medium,tavily,4.50,4.25,2423.06
7,medium,brave_search,4.25,3.00,978.72
6,medium,brave_answers,3.75,3.25,16592.12


In [6]:
# Decision 1: Do we need web follow-up at all?
# If static committee is aligned + high confidence, skip web. Otherwise use web.

def should_use_web(static_action: str, static_conf: float, skeptic_verdict: str, skeptic_conf: float, risk_flag: bool) -> dict:
    disagreement = skeptic_verdict in {"DISAGREE", "MODIFY"}
    low_conf = static_conf < 55 or skeptic_conf < 6
    material_risk = bool(risk_flag)

    use_web = disagreement or low_conf or material_risk
    reason = []
    if disagreement:
        reason.append("skeptic disagreement")
    if low_conf:
        reason.append("low confidence")
    if material_risk:
        reason.append("elevated risk flag")
    if not reason:
        reason = ["committee aligned with high confidence"]

    return {"use_web": use_web, "reason": ", ".join(reason)}


# Decision 2: single API call vs mini research loop
# Rule baseline: easy->single, medium->single unless low quality, hard->mini.

def classify_research_action(difficulty: str, expected_action: str, provider_quality: float) -> str:
    if expected_action == "mini_research":
        return "mini_research"
    if difficulty == "hard":
        return "mini_research"
    if difficulty == "medium" and provider_quality < 3.5:
        return "mini_research"
    return "single_api_call"


# Provider routing by difficulty from measured performance
best_by_diff = (
    provider_diff.sort_values(["difficulty", "overall_quality", "latency_ms"], ascending=[True, False, True])
    .groupby("difficulty", as_index=False)
    .first()
)

print("Best provider by difficulty:")
display(best_by_diff.round(2))

routing_map = {r["difficulty"]: r["provider"] for _, r in best_by_diff.iterrows()}
print("Routing map:", routing_map)

Best provider by difficulty:


,difficulty,provider,overall_quality,truth_alignment,latency_ms
0,easy,brave_search,4.50,4.00,1010.50
1,hard,tavily,4.25,3.25,3111.46
2,medium,tavily,4.50,4.25,2423.06


Routing map: {'easy': 'brave_search', 'hard': 'tavily', 'medium': 'tavily'}


In [7]:
# Architecture + settings/cost/latency plan

if SETTINGS_AVAILABLE:
    per_member_caps = settings.research_max_calls_per_member_per_cycle
    total_cap = settings.research_max_total_calls_per_cycle
    monthly_caps = {
        "brave_search": settings.brave_search_monthly_calls,
        "brave_answers": settings.brave_answer_monthly_calls,
        "tavily": settings.tavily_monthly_calls,
    }
else:
    per_member_caps = {"strategy": 20, "skeptic": 8, "risk": 7}
    total_cap = 35
    monthly_caps = {"brave_search": 2000, "brave_answers": 2000, "tavily": 1000}

cost_per_call = {
    "brave_search": 0.005,
    "brave_answers": 0.004,  # excluding tokens
    "tavily": 0.008,
}

provider_rollup = scores_df.groupby("provider", as_index=False).agg(
    avg_quality=("overall_quality", "mean"),
    avg_truth=("truth_alignment", "mean"),
    avg_latency_ms=("latency_ms", "mean"),
)

policy_rows = []
for _, r in provider_rollup.iterrows():
    p = r["provider"]
    # Utility: quality-heavy, latency penalty, light cost penalty
    utility = (r["avg_quality"] * 0.55) + (r["avg_truth"] * 0.25) - (r["avg_latency_ms"] / 10000) - (cost_per_call.get(p, 0) * 10)
    policy_rows.append({
        "provider": p,
        "avg_quality": round(float(r["avg_quality"]), 2),
        "avg_truth": round(float(r["avg_truth"]), 2),
        "avg_latency_ms": round(float(r["avg_latency_ms"]), 1),
        "cost_per_call_usd": cost_per_call.get(p, 0),
        "utility": round(float(utility), 3),
        "monthly_cap": monthly_caps.get(p),
    })

policy_df = pd.DataFrame(policy_rows).sort_values("utility", ascending=False)

architecture_plan = {
    "gating": {
        "use_web_if": [
            "skeptic verdict is DISAGREE or MODIFY",
            "strategy conviction < 55 or skeptic confidence < 6",
            "risk flag true",
        ],
        "otherwise": "skip web and keep static decision",
    },
    "action_mode": {
        "easy": "single_api_call",
        "medium": "single_api_call (escalate to mini if quality low)",
        "hard": "mini_research",
    },
    "tool_budget": {
        "per_member": per_member_caps,
        "max_total_per_cycle": total_cap,
    },
    "provider_routing_by_difficulty": routing_map,
    "provider_rank_overall": policy_df["provider"].tolist(),
    "notes": [
        "Gemini moderation stage currently single-turn (no tool loop in implementation).",
        "Skeptic GPT tool-use loop exists and prompt advises sparse usage (1-2 searches).",
        "Strategy Claude tool-use loop supports up to 8 iterations with research timeout guard.",
    ],
}

print("Provider utility table:")
display(policy_df)

print("\nArchitecture plan draft:")
print(json.dumps(architecture_plan, indent=2))

Provider utility table:


,provider,avg_quality,avg_truth,avg_latency_ms,cost_per_call_usd,utility,monthly_cap
1,brave_search,4.17,2.83,992.2,0.005,2.851,2000
2,tavily,4.17,3.42,2578.2,0.008,2.808,1000
0,brave_answers,3.42,2.92,22663.0,0.004,0.302,2000



Architecture plan draft:
{
  "gating": {
    "use_web_if": [
      "skeptic verdict is DISAGREE or MODIFY",
      "strategy conviction < 55 or skeptic confidence < 6",
      "risk flag true"
    ],
    "otherwise": "skip web and keep static decision"
  },
  "action_mode": {
    "easy": "single_api_call",
    "medium": "single_api_call (escalate to mini if quality low)",
    "hard": "mini_research"
  },
  "tool_budget": {
    "per_member": {
      "strategy": 20,
      "skeptic": 8,
      "risk": 7
    },
    "max_total_per_cycle": 35
  },
  "provider_routing_by_difficulty": {
    "easy": "brave_search",
    "hard": "tavily",
    "medium": "tavily"
  },
  "provider_rank_overall": [
    "brave_search",
    "tavily",
    "brave_answers"
  ],
  "notes": [
    "Gemini moderation stage currently single-turn (no tool loop in implementation).",
    "Skeptic GPT tool-use loop exists and prompt advises sparse usage (1-2 searches).",
    "Strategy Claude tool-use loop supports up to 8 iterations

In [8]:
# End-to-end ticker test (CB): static vs gated web-enriched workflow

CB = "CB_US_EQ"

# Simulated static committee outputs (replace with live model calls if needed)
static_committee = {
    "strategy": {"action": "HOLD", "conviction": 45, "reasoning": "Mixed outlook and valuation uncertainty."},
    "skeptic": {"verdict": "DISAGREE", "confidence": 7, "reasoning": "Potential downside from guidance and valuation."},
    "risk": {"risk_flag": True, "reasoning": "Uncertain macro sensitivity."},
}

web_gate = should_use_web(
    static_action=static_committee["strategy"]["action"],
    static_conf=static_committee["strategy"]["conviction"],
    skeptic_verdict=static_committee["skeptic"]["verdict"],
    skeptic_conf=static_committee["skeptic"]["confidence"],
    risk_flag=static_committee["risk"]["risk_flag"],
)

print("CB static committee:")
print(json.dumps(static_committee, indent=2))
print("\nWeb gating decision:")
print(json.dumps(web_gate, indent=2))

# Build follow-up queue as if requested by LLM committee after static review
cb_followups = [q for q in QUESTION_SET if q["ticker"] in {"CB", "GENERAL"}]
followup_rows = []
for q in cb_followups:
    # choose provider by difficulty map
    provider = routing_map.get(q["difficulty"], policy_df.iloc[0]["provider"])
    mode = classify_research_action(q["difficulty"], q["expected_action"], float(policy_df[policy_df["provider"] == provider]["avg_quality"].iloc[0]))

    if web_gate["use_web"]:
        out = run_provider(provider, q["question"])
        followup_rows.append({
            "id": q["id"],
            "difficulty": q["difficulty"],
            "member": q["member"],
            "provider": provider,
            "action_mode": mode,
            "ok": out.get("ok", False),
            "latency_ms": round(float(out.get("latency_ms", 0)), 2),
            "num_results": out.get("num_results", 0),
            "answer_preview": (out.get("answer") or "")[:140],
        })
    else:
        followup_rows.append({
            "id": q["id"],
            "difficulty": q["difficulty"],
            "member": q["member"],
            "provider": provider,
            "action_mode": mode,
            "ok": None,
            "latency_ms": 0,
            "num_results": 0,
            "answer_preview": "web skipped by gate",
        })

cb_run_df = pd.DataFrame(followup_rows)
print("\nCB end-to-end follow-up execution table:")
display(cb_run_df)

final_recommendation = {
    "ticker": CB,
    "web_gate_decision": web_gate,
    "followup_plan": cb_run_df.to_dict(orient="records"),
    "implementation_recommendation": {
        "default_mode": "static-first",
        "escalation": "web only when gate says material change likely",
        "easy_medium": "single API call",
        "hard": "mini research loop",
        "provider_routing_by_difficulty": routing_map,
        "budget": {
            "per_member": per_member_caps,
            "max_total": total_cap,
        },
    },
}

rec_path = PROJECT_ROOT / "data" / "research_policy_recommendation.json"
rec_path.write_text(json.dumps(final_recommendation, indent=2), encoding="utf-8")

print(f"Saved final recommendation to: {rec_path}")

CB static committee:
{
  "strategy": {
    "action": "HOLD",
    "conviction": 45,
    "reasoning": "Mixed outlook and valuation uncertainty."
  },
  "skeptic": {
    "verdict": "DISAGREE",
    "confidence": 7,
    "reasoning": "Potential downside from guidance and valuation."
  },
  "risk": {
    "risk_flag": true,
    "reasoning": "Uncertain macro sensitivity."
  }
}

Web gating decision:
{
  "use_web": true,
  "reason": "skeptic disagreement, low confidence, elevated risk flag"
}

CB end-to-end follow-up execution table:


,id,difficulty,member,provider,action_mode,ok,latency_ms,num_results,answer_preview
0,E4,easy,risk,brave_search,single_api_call,True,638.02,5,
1,M1,medium,skeptic,tavily,single_api_call,True,667.10,5,"Based on the provided data, it appears that Ar..."
2,M2,medium,skeptic,tavily,single_api_call,True,1977.47,5,Recent analyst downgrades or upgrades for CB (...
3,M3,medium,risk,tavily,single_api_call,True,337.19,5,"The 8-K filing, also known as SEC Form 8-K, is..."
4,M4,medium,strategy,tavily,single_api_call,True,342.13,5,Based on the query regarding the CB catastroph...
5,H1,hard,risk,tavily,mini_research,True,344.15,5,In response to the query regarding the event w...
6,H2,hard,skeptic,tavily,mini_research,True,339.84,5,To construct a bear-case chain based on the gi...
7,H3,hard,strategy,tavily,mini_research,True,354.85,5,To materially flip a HOLD to BUY or SELL in th...
8,H4,hard,strategy,tavily,mini_research,True,345.26,5,To build a filing timeline after the quarter-e...


Saved final recommendation to: /home/deploy_invest_ai/investment-agent/data/research_policy_recommendation.json


## Notes on production mapping

- This notebook is intentionally **static-first** and evaluates whether web changes decisions materially.
- In production, map this to orchestrator phases:
  - Strategy: web tools allowed when `research.enabled && strategy_research_enabled`
  - Skeptic (GPT): web tools allowed when `research.enabled && skeptic_research_enabled`
  - Risk (Gemini): currently single-turn in implementation; tool loop is TBD
- Canonical docs:
  - architecture/naming: `docs/AGENTIC_RESEARCH.md`
  - routing policy: `docs/FOLLOWUP_RESEARCH_ROUTING_PLAN.md`
- Cost/latency and utility ranking should be recalibrated weekly from fresh benchmark runs.